In [ ]:
!pip -q install --upgrade openai pandas tqdm

import os, time, base64, glob, re
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from google.colab import drive

# mount google drive
drive.mount('/content/drive')

# API Key
os.environ["DASHSCOPE_API_KEY"] = os.environ.get("DASHSCOPE_API_KEY", "API-KEY")

client = OpenAI(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope-intl.aliyuncs.com/compatible-mode/v1"
)

MODEL_NAME = "qwen3-vl-flash-2026-01-22"

# Path
INPUT_DIR = "/content/drive/MyDrive/Thesis/dataset"
OUTPUT_CSV_PATH = "/content/drive/MyDrive/Thesis/pseudo_label_output.csv"


# Rate limit & checkpoint
DELAY_SECONDS = 2.0
SAVE_EVERY = 10

# Prompt
prompt = r"""
Anda adalah anotator ahli dalam psikologi, dan ahli linguistik. Tugas Anda adalah mengklasifikasikan meme menjadi [SELF-HARM] atau [NON-SELF-HARM] dengan presisi tinggi.
Ikuti alur penalaran (reasoning) di bawah ini secara berurutan. Jangan melompati langkah.
Langkah 1: Analisis Multimodal & Relasi
Lakukan analisis mendalam terhadap input dengan menjawab pertanyaan berikut:
1.  Analisis visual : apa objek, aktivitas, atau simbol yang muncul di gambar? apakah ada luka, darah, alat berbahaya, atau ekspresi emosional tertentu?
2.  Analisis tekstual: apa makna harfiah dari teks? apakah ada kata kunci terkait kematian, rasa sakit, atau keputusasaan?
3.  Analisis relasi (PENTING), yaitu bagaimana teks dan gambar berinteraksi? Apakah teks mengubah makna gambar netral menjadi berbahaya? Apakah ini sindiran (sarkasme), dark humor, atau hanya  ungkapan biasa ? Identifikasi resiko gabungan yang tidak terlihat jika teks/gambar dipisah.

Langkah 2: Indikator
Berdasarkan analisis relasi di atas, periksa apakah meme ini memenuhi salah satu indikator valid berikut secara ketat:
A. Indikator Narasi & Teks
 (1a) Metode dan deskripsi, mendeskripsikan cara menyakiti diri (cutting, burning, overdosis)
 (1b) Masking & romantisasi, menganggap self-harm estetik, puitis, atau mekanisme coping yang wajar
(1c)  Ideasi bunuh diri, menyiratkan keinginan mati
(1d) Gaslighting/manipulasi, menyalahkan orang lain atas niat menyakiti diri sendiri.
B. Indikator visual dan simbolik
 (2a) Luka fisik,Gambar darah, bekas sayatan, lebam.
(2b) Alat berbahaya seperti Silet, pisau, tali gantung, tumpukan obat
 (2c) Estetika dark humor, gambar ceria/kartun imut tetapi teksnya tentang keinginan mati/bunuh diri (kontras ironis)
Instruksi Label:
 Jika “terdapat minimal satu” indikator terpenuhi (terutama indikator relasional 2c) -> Klasifikasikan sebagai [SELF-HARM].
 Jika tidak ada indikator terpenuhi dan konteks hanya humor umum -> Klasifikasikan sebagai [NON-SELF-HARM].

Langkah 3: Self-checking
Sebelum memberikan label final, kritik keputusan Anda sendiri:
 "Apakah saya salah mengartikan *slang* atau lelucon internet sebagai ancaman serius?"
 "Apakah saya mengabaikan sinyal bahaya terselubung karena gambarnya terlihat lucu (kartun)?"
 Jika ragu, prioritaskan keselamatan (safety-first approach) namun tetap bedakan antara dark jokes  umum vs promosi self-harm spesifik.
"""

# Helpers
def encode_image_to_data_url(image_path: str) -> str:
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")

    ext = os.path.splitext(image_path.lower())[1]
    if ext == ".png":
        mime = "image/png"
    elif ext in [".jpg", ".jpeg"]:
        mime = "image/jpeg"
    elif ext == ".webp":
        mime = "image/webp"
    else:
        mime = "application/octet-stream"

    return f"data:{mime};base64,{encoded}"

def run_llm_on_image(image_path: str) -> str:
    image_data_url = encode_image_to_data_url(image_path)

    completion = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "image_url", "image_url": {"url": image_data_url}},
                    {"type": "text", "text": prompt},
                ],
            }
        ],
        stream=False,
    )

    return completion.choices[0].message.content

def safe_save_csv(df: pd.DataFrame, out_path: str):
    """Atomic write biar CSV gak korup kalau runtime mati pas nulis."""
    tmp_path = out_path + ".tmp"
    df.to_csv(tmp_path, index=False)
    os.replace(tmp_path, out_path)

def parse_llm_output(text: str) -> dict:
    """
    Ambil field dari output LLM.
    Kalau format agak melenceng, tetap berusaha ambil yang ada.
    """
    def grab(patterns, default=""):
        for pat in patterns:
            m = re.search(pat, text, flags=re.IGNORECASE | re.DOTALL)
            if m:
                return m.group(1).strip()
        return default

    teks_terlihat = grab([
        r"Teks\s*Terlihat\s*:\s*(.*?)(?=\n\s*(Label|Confidance|Confidence|Relasi)\s*:|\Z)"
    ])

    label = grab([
        r"Label\s*:\s*(SELF-HARM|NON\s*SELF\s*HARM|NON-SELF-HARM|NONSELFHARM|SELFHARM)"
    ])
    label = label.replace(" ", "").replace("_", "-").upper()
    if label == "NONSELFHARM":
        label = "NON-SELF-HARM"
    if label == "SELFHARM":
        label = "SELF-HARM"

    conf = grab([
        r"Confidance\s*:\s*([0-9]{1,3}\s*%?)",
        r"Confidence\s*:\s*([0-9]{1,3}\s*%?)"
    ])
    conf = conf.replace(" ", "")
    if conf and not conf.endswith("%"):
        # jika model ngasih "92" tanpa %
        conf = conf + "%"

    relasi = grab([
        r"Relasi\s*(?:gambar\s*dan\s*teks)?\s*:\s*(.*?)(?=\n\s*\w+\s*:|\Z)"
    ])

    return {
        "Teks Terlihat": teks_terlihat,
        "Label": label,
        "Confidance": conf,
        "Relasi Gambar dan Teks": relasi
    }

# Collect image files
image_exts = ("*.png", "*.jpg", "*.jpeg", "*.webp")
image_paths = []
for pat in image_exts:
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, pat)))

image_paths = sorted(image_paths)
print("Jumlah gambar ditemukan:", len(image_paths))
if len(image_paths) == 0:
    raise ValueError(f"Tidak ada gambar di folder: {INPUT_DIR}")


# Resume
done_set = set()
rows = []

if os.path.exists(OUTPUT_CSV_PATH):
    try:
        old = pd.read_csv(OUTPUT_CSV_PATH)
        # simpan filepath internal di CSV untuk resume
        if "filepath" in old.columns:
            done_set = set(old["filepath"].astype(str).tolist())
        rows = old.to_dict("records")
        print(f"Resume aktif: sudah ada {len(done_set)} item di CSV lama.")
    except Exception as e:
        print("Gagal baca CSV lama, lanjut dari nol. Error:", e)

processed_since_save = 0

# Main loop
for img_path in tqdm(image_paths):
    if img_path in done_set:
        continue

    fname = os.path.basename(img_path)

    try:
        raw_out = run_llm_on_image(img_path)
        parsed = parse_llm_output(raw_out)
        row = {
            "filename": fname,
            "Teks Terlihat": parsed["Teks Terlihat"],
            "Label": parsed["Label"],
            "Confidance": parsed["Confidance"],
            "Relasi Gambar dan Teks": parsed["Relasi Gambar dan Teks"],
            "filepath": img_path
        }
    except Exception as e:
        row = {
            "filename": fname,
            "Teks Terlihat": "",
            "Label": "",
            "Confidance": "",
            "Relasi Gambar dan Teks": f"[ERROR] {type(e).__name__}: {e}",
            "filepath": img_path
        }

    rows.append(row)
    done_set.add(img_path)
    processed_since_save += 1

    # Save
    if processed_since_save >= SAVE_EVERY:
        df = pd.DataFrame(rows)
        cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
        df = df.reindex(columns=cols)
        safe_save_csv(df, OUTPUT_CSV_PATH)
        processed_since_save = 0

    time.sleep(DELAY_SECONDS)

# Final save
df = pd.DataFrame(rows)
cols = ["filename", "Teks Terlihat", "Label", "Confidance", "Relasi Gambar dan Teks", "filepath"]
df = df.reindex(columns=cols)
safe_save_csv(df, OUTPUT_CSV_PATH)

print("Selesai! CSV tersimpan di:", OUTPUT_CSV_PATH)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Jumlah gambar ditemukan: 2183


100%|██████████| 2183/2183 [2:00:30<00:00,  3.31s/it]

Selesai! CSV tersimpan di: /content/drive/MyDrive/Thesis/pseudo_label_output.csv
